# Hate Speech BERT Classification - Binary (M3 Optimized)

This notebook trains a BERT model for **binary classification** of hate speech, optimized for local execution on Apple Silicon (M3).

Unlike the 3-class version, we filter out 'offensive' samples and only classify:
- **hatespeech** (0): Content that promotes violence or discrimination against protected groups
- **normal** (1): Non-hateful content

## Prerequisites & Setup

Before running this notebook, ensure you have the following set up:

1. **Python Environment**: Create and activate a virtual environment:
    ```bash
    python3 -m venv .venv-bert
    source .venv/bin/activate
    ```
2. **Install Dependencies**:
    ```bash
    pip install torch transformers datasets evaluate accelerate scikit-learn pandas tqdm ipykernel
    ```
3. **Register Kernel** (if using Jupyter):
    ```bash
    python -m ipykernel install --user --name=.venv
    ```
4. **Local Dataset**: Ensure the `hatexplain/` folder exists with JSONL files, or the notebook will download from HuggingFace.

In [ ]:
import torch
import numpy as np
import pandas as pd
import json
import os
import gc
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

## 1. Device Setup (M3 Optimized)

We detect and use Apple Silicon MPS acceleration when available.

In [2]:
def get_device():
    """Get the best available device for M3 Mac."""
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"✅ Using GPU: {torch.cuda.get_device_name(0)}")
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        device = torch.device("mps")
        print("✅ Using Apple Silicon MPS acceleration")
    else:
        device = torch.device("cpu")
        print("⚠️ Using CPU (training will be slower)")
    return device

device = get_device()
print(f"PyTorch version: {torch.__version__}")

✅ Using Apple Silicon MPS acceleration
PyTorch version: 2.10.0


## 2. Binary Label Configuration

We use only 2 labels: hatespeech (0) and normal (1).

In [3]:
# BINARY classification: hatespeech vs normal
ID2NAME = {0: "hatespeech", 1: "normal"}
NAME2ID = {v: k for k, v in ID2NAME.items()}

# Original 3-class mapping (for reference during data filtering)
ORIGINAL_ID2NAME = {0: "hatespeech", 1: "normal", 2: "offensive"}

print(f"Binary classification labels: {ID2NAME}")
print(f"Number of classes: {len(ID2NAME)}")

Binary classification labels: {0: 'hatespeech', 1: 'normal'}
Number of classes: 2


## 3. Load Dataset

Load from local JSONL files or HuggingFace Hub.

In [4]:
print("Loading HateXplain dataset...")

data_files = {
    "train": "hatexplain/hatexplain_train.jsonl",
    "validation": "hatexplain/hatexplain_validation.jsonl",
    "test": "hatexplain/hatexplain_test.jsonl"
}

local_files_exist = all(os.path.exists(path) for path in data_files.values())

if local_files_exist:
    print("✅ Loading from local JSONL files...")
    ds = load_dataset("json", data_files=data_files)
else:
    print("⚠️ Local files not found. Loading from HuggingFace Hub...")
    ds = load_dataset("hatexplain")

print(ds)

Loading HateXplain dataset...
✅ Loading from local JSONL files...
DatasetDict({
    train: Dataset({
        features: ['id', 'annotators', 'rationales', 'post_tokens'],
        num_rows: 15383
    })
    validation: Dataset({
        features: ['id', 'annotators', 'rationales', 'post_tokens'],
        num_rows: 1922
    })
    test: Dataset({
        features: ['id', 'annotators', 'rationales', 'post_tokens'],
        num_rows: 1924
    })
})


In [5]:
# Data Normalization Functions
def _coerce_annotators(ann):
    """Parse annotators from various formats."""
    if ann is None:
        return []
    if isinstance(ann, str):
        try:
            ann = json.loads(ann)
        except Exception:
            return []
    if isinstance(ann, list):
        if len(ann) == 0:
            return []
        if isinstance(ann[0], dict):
            return ann
        out = []
        for x in ann:
            try:
                out.append({"label": int(x), "target": []})
            except Exception:
                pass
        return out
    if isinstance(ann, dict):
        labels = ann.get("label", [])
        targets = ann.get("target", [])
        n = len(labels)
        out = []
        for i in range(n):
            try:
                lab = int(labels[i])
            except Exception:
                continue
            tgt = targets[i] if (isinstance(targets, list) and i < len(targets)) else []
            out.append({"label": lab, "target": tgt if isinstance(tgt, list) else [tgt]})
        return out
    return []


def _majority_vote_label(anns):
    """Majority vote with tie-breaker: hate > offensive > normal."""
    pri = [0, 2, 1]
    counts = {0: 0, 1: 0, 2: 0}
    for a in anns:
        lab = int(a.get("label", 1))
        if lab in counts:
            counts[lab] += 1
    if counts[0] == counts[1] == counts[2] == 0:
        return 1
    return sorted(counts.items(), key=lambda kv: (-kv[1], pri.index(kv[0])))[0][0]


def _pick_text(ex):
    """Extract text from example."""
    if "post_tokens" in ex and isinstance(ex["post_tokens"], list):
        return " ".join(ex["post_tokens"])
    if "text" in ex:
        return str(ex["text"])
    return ""


def normalize_example(e):
    """Normalize a single example."""
    text = _pick_text(e)
    anns = _coerce_annotators(e.get("annotators", None))
    label = _majority_vote_label(anns)
    
    return {
        "text_norm": text.strip(),
        "label_original": int(label),
        "label_name_original": ORIGINAL_ID2NAME[label],
    }

In [6]:
print("Normalizing dataset...")
ds = ds.map(normalize_example)

# Keep only needed columns
to_keep = ["text_norm", "label_original", "label_name_original"]
drop_cols = [c for c in ds["train"].column_names if c not in to_keep]
ds = ds.remove_columns(drop_cols)

# Drop empty text
ds = ds.filter(lambda t: len(t.strip()) > 0, input_columns=["text_norm"])

print("\nBefore filtering (3-class distribution):")
for split in ds:
    labels = ds[split]["label_original"]
    counts = {ORIGINAL_ID2NAME[k]: labels.count(k) for k in [0, 1, 2]}
    print(f"  {split}: {counts}")

Normalizing dataset...


Filter: 100%|██████████| 1924/1924 [00:00<00:00, 768995.70 examples/s]



Before filtering (3-class distribution):
  train: {'hatespeech': 4748, 'normal': 6251, 'offensive': 4384}
  validation: {'hatespeech': 593, 'normal': 781, 'offensive': 548}
  test: {'hatespeech': 594, 'normal': 782, 'offensive': 548}


In [7]:
print("Filtering out 'offensive' for binary classification...")

# Filter out offensive (label_original == 2)
ds_binary = ds.filter(lambda x: x["label_original"] != 2)

# Add binary label column
def add_binary_label(example):
    return {
        "label_norm": example["label_original"],
        "label_name": ID2NAME[example["label_original"]],
    }

ds_binary = ds_binary.map(add_binary_label)

print("\n✅ After filtering (binary distribution):")
for split in ds_binary:
    labels = ds_binary[split]["label_norm"]
    counts = {ID2NAME[k]: labels.count(k) for k in [0, 1]}
    total = len(ds_binary[split])
    print(f"  {split}: {counts} (total: {total})")

Filtering out 'offensive' for binary classification...


Map: 100%|██████████| 1376/1376 [00:00<00:00, 42017.85 examples/s]


✅ After filtering (binary distribution):
  train: {'hatespeech': 4748, 'normal': 6251} (total: 10999)
  validation: {'hatespeech': 593, 'normal': 781} (total: 1374)
  test: {'hatespeech': 594, 'normal': 782} (total: 1376)


## 4. Initialize Model

In [30]:
num_labels = len(ID2NAME)  # 2

print(f"Loading {MODEL_NAME} with {num_labels} labels...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
)

model.config.id2label = ID2NAME
model.config.label2id = NAME2ID
model.to(device)

print(f"✅ Model loaded on {device}")

Loading bert-base-uncased with 2 labels...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1889.12it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those p

✅ Model loaded on mps


## 5. Tokenization

In [31]:
def tokenize_fn(batch):
    enc = tokenizer(
        batch["text_norm"], 
        truncation=True, 
        padding=False, 
        max_length=MAX_LEN
    )
    enc["labels"] = batch["label_norm"]
    return enc

print("Tokenizing dataset...")
tok_ds = ds_binary.map(
    tokenize_fn, 
    batched=True, 
    remove_columns=ds_binary["train"].column_names
)

collator = DataCollatorWithPadding(tokenizer)
print("✅ Tokenization complete!")

Tokenizing dataset...


Map: 100%|██████████| 1374/1374 [00:00<00:00, 22162.39 examples/s]

✅ Tokenization complete!


## 6. Metrics & Weighted Trainer

In [32]:
acc = evaluate.load("accuracy")
f1 = evaluate.load("f1")
prec = evaluate.load("precision")
rec = evaluate.load("recall")

def compute_metrics(eval_pred):
    """Compute metrics for binary classification."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    
    out = {
        "accuracy": acc.compute(predictions=preds, references=labels)["accuracy"],
        "macro_f1": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
        "macro_precision": prec.compute(predictions=preds, references=labels, average="macro")["precision"],
        "macro_recall": rec.compute(predictions=preds, references=labels, average="macro")["recall"],
    }
    
    labels_np = np.array(labels)
    for i, name in ID2NAME.items():
        out[f"f1_{name}"] = f1.compute(
            predictions=(preds == i).astype(int),
            references=(labels_np == i).astype(int),
            average="binary"
        )["f1"]
    
    return out

print("✅ Metrics loaded")

✅ Metrics loaded


In [33]:
import torch.nn as nn

# Compute class weights
counts = np.bincount(np.array(ds_binary["train"]["label_norm"]))
class_weights = (1.0 / np.clip(counts, 1, None))
class_weights = class_weights / class_weights.sum() * len(class_weights)
print(f"Class counts: {counts}")
print(f"Class weights: {class_weights}")


class WeightedCELoss(nn.Module):
    def __init__(self, weights=None):
        super().__init__()
        if weights is not None:
            self.weights = torch.tensor(weights, dtype=torch.float).to(device)
            self.loss = nn.CrossEntropyLoss(weight=self.weights)
        else:
            self.loss = nn.CrossEntropyLoss()
    
    def forward(self, logits, labels):
        return self.loss(logits, labels)


class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.criterion = (
            WeightedCELoss(class_weights) 
            if class_weights is not None 
            else nn.CrossEntropyLoss()
        )
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = self.criterion(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

Class counts: [4748 6251]
Class weights: [1.13664879 0.86335121]


## 7. Training

In [41]:
# --- Configuration ---
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128
BATCH_SIZE_TRAIN = 12
BATCH_SIZE_EVAL = 32
NUM_EPOCHS = 5
LEARNING_RATE = 2e-6
SAVE_PATH = "./bert-binary-hatexplain-finetuned"

In [42]:
# M3-optimized training arguments
from transformers import EarlyStoppingCallback
# Add early stopping after 2 epochs without improvement
early_stop = EarlyStoppingCallback(early_stopping_patience=2)

training_args = TrainingArguments(
    output_dir="./runs/bert-binary-hatexplain",
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE_TRAIN,
    per_device_eval_batch_size=BATCH_SIZE_EVAL,
    weight_decay=0.3,
    warmup_steps=100,
    lr_scheduler_type="linear",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    # MPS doesn't support fp16, so only enable on CUDA
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to="none",
    optim="adamw_torch",
)

print("✅ Training arguments configured")

✅ Training arguments configured


In [43]:
print("🔄 Reloading fresh model weights...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
)
model.config.id2label = ID2NAME
model.config.label2id = NAME2ID
model.to(device)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tok_ds["train"],
    eval_dataset=tok_ds["validation"],
    # tokenizer=tokenizer,
    callbacks=[early_stop],
    data_collator=collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

print("🚀 Starting training...")
trainer.train()

🔄 Reloading fresh model weights...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1580.37it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those p

🚀 Starting training...


/Users/shariff/Documents/github/11th-DAP-hate-speech/.venv-bert/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Macro Precision,Macro Recall,F1 Hatespeech,F1 Normal
1,0.407982,0.400100,0.842795,0.838886,0.841257,0.837157,0.813793,0.863980
2,0.336321,0.373735,0.855895,0.852712,0.853886,0.851725,0.831058,0.874365
3,0.308500,0.368939,0.862445,0.859376,0.860670,0.858299,0.838599,0.880152
4,0.277817,0.365433,0.863901,0.861657,0.860851,0.862624,0.844037,0.879277
5,0.267916,0.362022,0.867540,0.865013,0.865013,0.865013,0.846543,0.883483


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.36it/s]
/Users/shariff/Documents/github/11th-DAP-hate-speech/.venv-bert/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.08it/s]
/Users/shariff/Documents/github/11th-DAP-hate-speech/.venv-bert/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.55it/s]
/Users/shariff/Documents/github/11th-DAP-hate-speech/.venv-bert/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super

TrainOutput(global_step=4585, training_loss=0.3604348942913692, metrics={'train_runtime': 2304.0932, 'train_samples_per_second': 23.868, 'train_steps_per_second': 1.99, 'total_flos': 1656504551777820.0, 'train_loss': 0.3604348942913692, 'epoch': 5.0})

## 8. Evaluation

In [44]:
print("Evaluating on test set...")
test_metrics = trainer.evaluate(tok_ds["test"])

print("\n" + "="*50)
print("TEST RESULTS")
print("="*50)
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")

Evaluating on test set...


/Users/shariff/Documents/github/11th-DAP-hate-speech/.venv-bert/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



TEST RESULTS
eval_loss: 0.3401
eval_accuracy: 0.8765
eval_macro_f1: 0.8740
eval_macro_precision: 0.8743
eval_macro_recall: 0.8737
eval_f1_hatespeech: 0.8564
eval_f1_normal: 0.8916
eval_runtime: 9.3268
eval_samples_per_second: 147.5310
eval_steps_per_second: 4.6100
epoch: 5.0000


In [45]:
from sklearn.metrics import classification_report, confusion_matrix

predictions = trainer.predict(tok_ds["test"])
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print("Classification Report:\n")
print(classification_report(labels, preds, target_names=[ID2NAME[0], ID2NAME[1]]))

print("\nConfusion Matrix:")
cm = confusion_matrix(labels, preds, labels=[0, 1])
print(f"                 Predicted")
print(f"                 hate    normal")
print(f"Actual hate    | {cm[0][0]:5d}   {cm[0][1]:5d}")
print(f"       normal  | {cm[1][0]:5d}   {cm[1][1]:5d}")

/Users/shariff/Documents/github/11th-DAP-hate-speech/.venv-bert/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Classification Report:

              precision    recall  f1-score   support

  hatespeech       0.86      0.85      0.86       594
      normal       0.89      0.89      0.89       782

    accuracy                           0.88      1376
   macro avg       0.87      0.87      0.87      1376
weighted avg       0.88      0.88      0.88      1376


Confusion Matrix:
                 Predicted
                 hate    normal
Actual hate    |   507      87
       normal  |    83     699


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from datetime import datetime
import csv

predictions = trainer.predict(tok_ds["test"])
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Get classification report as dict for structured access
report_dict = classification_report(labels, preds, target_names=[ID2NAME[0], ID2NAME[1]], output_dict=True)
cm = confusion_matrix(labels, preds, labels=[0, 1])

# --- Pretty Print Results ---
print("\n" + "═" * 60)
print("  📊 CLASSIFICATION RESULTS")
print("═" * 60)

# Overall Metrics
print("\n┌" + "─" * 58 + "┐")
print("│" + " OVERALL METRICS".center(58) + "│")
print("├" + "─" * 29 + "┬" + "─" * 28 + "┤")
print(f"│  Accuracy:  {report_dict['accuracy']*100:>6.2f}%" + " " * 12 + "│" + f"  Macro F1:  {report_dict['macro avg']['f1-score']*100:>6.2f}%" + " " * 6 + "│")
print("└" + "─" * 29 + "┴" + "─" * 28 + "┘")

# Per-Class Metrics
print("\n┌" + "─" * 58 + "┐")
print("│" + " PER-CLASS METRICS".center(58) + "│")
print("├" + "─" * 14 + "┬" + "─" * 14 + "┬" + "─" * 14 + "┬" + "─" * 13 + "┤")
print("│" + "  Class".center(14) + "│" + "  Precision".center(14) + "│" + "  Recall".center(14) + "│" + "  F1-Score".center(13) + "│")
print("├" + "─" * 14 + "┼" + "─" * 14 + "┼" + "─" * 14 + "┼" + "─" * 13 + "┤")
for class_name in [ID2NAME[0], ID2NAME[1]]:
    metrics = report_dict[class_name]
    print(f"│  {class_name:<11} │" + f"  {metrics['precision']*100:>6.2f}%" + "    │" + f"  {metrics['recall']*100:>6.2f}%" + "    │" + f"  {metrics['f1-score']*100:>5.2f}%" + "   │")
print("└" + "─" * 14 + "┴" + "─" * 14 + "┴" + "─" * 14 + "┴" + "─" * 13 + "┘")

# Confusion Matrix
print("\n┌" + "─" * 38 + "┐")
print("│" + " CONFUSION MATRIX".center(38) + "│")
print("├" + "─" * 38 + "┤")
print("│" + "              Predicted".ljust(38) + "│")
print("│" + "           hatespeech   normal".ljust(38) + "│")
print("├" + "─" * 38 + "┤")
print(f"│  Actual hatespeech │  {cm[0][0]:>5}   │  {cm[0][1]:>5}   │")
print(f"│         normal     │  {cm[1][0]:>5}   │  {cm[1][1]:>5}   │")
print("└" + "─" * 38 + "┘")

# --- Save Results to CSV ---
results_dir = "results"
os.makedirs(results_dir, exist_ok=True)
results_file = os.path.join(results_dir, "model_scores.csv")

# Prepare result entry
result_entry = {
    "timestamp": datetime.now().isoformat(),
    "model_name": MODEL_NAME,
    "model_type": "trained encoder", 
    "task": "binary",
    "epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE_TRAIN,
    "learning_rate": LEARNING_RATE,
    "max_length": MAX_LEN,
    "accuracy": round(report_dict["accuracy"], 4),
    "macro_f1": round(report_dict["macro avg"]["f1-score"], 4),
    "macro_precision": round(report_dict["macro avg"]["precision"], 4),
    "macro_recall": round(report_dict["macro avg"]["recall"], 4),
    "hatespeech_f1": round(report_dict[ID2NAME[0]]["f1-score"], 4),
    "hatespeech_precision": round(report_dict[ID2NAME[0]]["precision"], 4),
    "hatespeech_recall": round(report_dict[ID2NAME[0]]["recall"], 4),
    "normal_f1": round(report_dict[ID2NAME[1]]["f1-score"], 4),
    "normal_precision": round(report_dict[ID2NAME[1]]["precision"], 4),
    "normal_recall": round(report_dict[ID2NAME[1]]["recall"], 4),
    "cm_hate_as_hate": int(cm[0][0]),
    "cm_hate_as_normal": int(cm[0][1]),
    "cm_normal_as_hate": int(cm[1][0]),
    "cm_normal_as_normal": int(cm[1][1]),
    "test_samples": len(labels),
}

# Check if file exists to determine if we need to write headers
file_exists = os.path.exists(results_file)

with open(results_file, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=result_entry.keys())
    if not file_exists:
        writer.writeheader()
    writer.writerow(result_entry)

print("\n" + "═" * 60)
print(f"  ✅ Results saved to: {results_file}")
print("═" * 60)

/Users/shariff/Documents/github/11th-DAP-hate-speech/.venv-bert/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



════════════════════════════════════════════════════════════
  📊 CLASSIFICATION RESULTS
════════════════════════════════════════════════════════════

┌──────────────────────────────────────────────────────────┐
│                      OVERALL METRICS                     │
├─────────────────────────────┬────────────────────────────┤
│  Accuracy:   87.65%            │  Macro F1:   87.40%      │
└─────────────────────────────┴────────────────────────────┘

┌──────────────────────────────────────────────────────────┐
│                     PER-CLASS METRICS                    │
├──────────────┬──────────────┬──────────────┬─────────────┤
│     Class    │   Precision  │     Recall   │    F1-Score │
├──────────────┼──────────────┼──────────────┼─────────────┤
│  hatespeech  │   85.93%    │   85.35%    │  85.64%   │
│  normal      │   88.93%    │   89.39%    │  89.16%   │
└──────────────┴──────────────┴──────────────┴─────────────┘

┌──────────────────────────────────────┐
│           CONFUSIO

## 9. Save Model

In [47]:
print(f"Saving model to {SAVE_PATH}...")
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print("✅ Model saved!")

Saving model to ./bert-binary-hatexplain-finetuned...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.68it/s]

✅ Model saved!


## 10. Inference Example

In [53]:
def predict_single(text):
    """Predict class for a single text."""
    model.eval()
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        truncation=True, 
        max_length=MAX_LEN
    ).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
        pred = torch.argmax(probs, dim=-1).item()
    
    return {
        "prediction": ID2NAME[pred],
        "confidence": probs[0][pred].item(),
        "prob_hatespeech": probs[0][0].item(),
        "prob_normal": probs[0][1].item(),
    }

# Test
test_texts = [
    "You look absolutely horrid today",
    "Fuck you shithead, stupid asian whore",
]

print("Testing inference:")
for text in test_texts:
    result = predict_single(text)
    print(f"\nText: {text}")
    print(f"  → {result['prediction']} ({result['confidence']:.1%} confident)")

Testing inference:

Text: You look absolutely horrid today
  → normal (89.7% confident)

Text: Fuck you shithead, stupid asian whore
  → hatespeech (96.2% confident)
